# 09 — Active Cache and Data Subscriptions

Pull model: `data_formula` with `_cache_time` for periodic refresh, `data.subscribe` for side effects.

**What you learn:**
- `data_formula` with `_cache_time=-N`: active cache with background refresh (requires async)
- `data.subscribe()`: react to data changes with side effects (logging, file writes)
- Pull model: formulas compute on-demand, active cache pushes periodic updates
- Side effects belong to the manager/application, not to the builder

**Prerequisites:** 08_reactive_basics

In [ ]:
import random
from pathlib import Path
from IPython.display import display, HTML, update_display

from genro_builders.builder import component
from genro_builders.contrib.html import HtmlBuilder
from genro_builders.manager import ReactiveManager
from genro_toolbox import smartawait

CSS = Path('dashboard.css').read_text()

In [ ]:
class DashBuilder(HtmlBuilder):
    @component(main_tag='div')
    def gauge(self, comp, label_text=None, value=None, unit=None, **kwargs):
        comp.div(label_text, _class='label')
        comp.div(value, _class='value')
        comp.div(unit, _class='unit')

class Dashboard(ReactiveManager):
    def __init__(self):
        self.page = self.set_builder('page', DashBuilder)

    def store(self, data):
        from genro_bag import Bag
        gauges = Bag()
        gauges['speed'] = 87
        gauges['rpm'] = 3250
        gauges['oil_temp'] = 92
        gauges['fuel'] = 73
        data['gauges'] = gauges

    def main(self, source):
        source.head().style(CSS)
        body = source.body()
        body.h1('DASHBOARD')
        dash = body.div(_class='dash', datapath='gauges')

        dash.gauge(_class='gauge speed', label_text='SPEED', value='^.speed', unit='km/h')
        dash.gauge(_class='gauge rpm', label_text='RPM', value='^.rpm', unit='x1000')
        dash.gauge(_class='gauge oil', label_text='OIL TEMP', value='^.oil_temp', unit='C')
        dash.gauge(_class='gauge fuel', label_text='FUEL', value='^.fuel', unit='%')

In [ ]:
app = Dashboard()
await smartawait(app.run(subscribe=True))

# Side effect: log gauge changes via data.subscribe
app.reactive_store.subscribe(
    'logger',
    any=lambda pathlist=None, **kw: print(f"  changed: {'.'.join(str(p) for p in pathlist)}") if pathlist else None,
)

HTML(app.page.render())

## Update gauges manually

In pull model, update data and re-render explicitly.

In [ ]:
store = app.reactive_store
store['gauges.speed'] = 120
store['gauges.rpm'] = 4500
print(f"Speed: {store['gauges.speed']}, RPM: {store['gauges.rpm']}")
HTML(app.page.render())